# DSPy Multimodal Support

This notebook demonstrates DSPy 3.x's multimodal capabilities for working with images and audio.

**What You'll Learn:**
- How to use `dspy.Image` in signatures and modules
- How to use `dspy.Audio` in signatures and modules
- How to build vision-language applications
- How to build speech/audio processing applications
- How to combine multimodal inputs with text

## Setup

For multimodal support, you need a vision/audio-capable model:
- **OpenAI**: gpt-4o (supports vision)
- **Anthropic**: claude-3-7-sonnet-20250219 (supports vision)
- **Google**: gemini-1.5-pro (supports vision and audio)

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import setup_default_lm, print_step, print_result, print_error, configure_dspy
from dotenv import load_dotenv

load_dotenv('../../.env')

try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=1000)
    configure_dspy(lm=lm)
    print_result("Language model configured successfully!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")
    print("Make sure you have set your OPENAI_API_KEY in the .env file")

## Part 1: Image Understanding

Use `dspy.Image` to analyze and understand images. DSPy supports loading images from file paths, URLs, or raw bytes.

In [ ]:
# Example 1.1: Image Description
print_step("Image Description", "Defining a signature for image analysis")

class ImageDescriber(dspy.Signature):
    """Describe an image in detail."""
    image = dspy.InputField(desc="An image to describe")
    description = dspy.OutputField(desc="A detailed description of the image")

describer = dspy.Predict(ImageDescriber)

print("Structure:")
print("  - Create a dspy.Image object from a file path or URL")
print("  - Pass it to the signature's image field")
print("  - The model will analyze and describe the image")
print("\nExample code:")
print("  img = dspy.Image(path='photo.jpg')  # or url='https://...'")
print("  result = describer(image=img)")
print("  print(result.description)")

In [ ]:
# Example 1.2: Visual Question Answering (VQA)
print_step("Visual Question Answering", "Answering questions about images")

class VisualQA(dspy.Signature):
    """Answer questions about an image."""
    image = dspy.InputField(desc="An image to analyze")
    question = dspy.InputField(desc="A question about the image")
    answer = dspy.OutputField(desc="The answer to the question based on the image")

vqa = dspy.Predict(VisualQA)

print("Use case: Answer specific questions about images")
print("Example:")
print("  img = dspy.Image(path='street_scene.jpg')")
print("  result = vqa(image=img, question='How many cars are visible?')")
print("  print(f'Answer: {result.answer}')")

In [ ]:
# Example 1.3: Image Comparison
print_step("Image Comparison", "Comparing multiple images")

class ImageComparison(dspy.Signature):
    """Compare two images and identify differences."""
    image1 = dspy.InputField(desc="First image")
    image2 = dspy.InputField(desc="Second image")
    differences = dspy.OutputField(desc="Key differences between the images")
    similarity_score = dspy.OutputField(desc="Similarity score from 0-10")

comparator = dspy.Predict(ImageComparison)

print("Use case: Find differences or similarities between images")
print("Example:")
print("  img1 = dspy.Image(path='before.jpg')")
print("  img2 = dspy.Image(path='after.jpg')")
print("  result = comparator(image1=img1, image2=img2)")
print("  print(f'Differences: {result.differences}')")
print("  print(f'Similarity: {result.similarity_score}/10')")

## Part 2: Image Generation

Use `dspy.Image` as an output field with models that support image generation.

In [ ]:
class ImageGenerator(dspy.Signature):
    """Generate an image based on a text description."""
    description = dspy.InputField(desc="Description of the image to generate")
    image = dspy.OutputField(desc="The generated image")

generator = dspy.Predict(ImageGenerator)

print("Use case: Generate images from text descriptions")
print("Note: This requires a model with image generation capabilities")
print("Example:")
print("  result = generator(description='A serene mountain lake at sunset')")
print("  result.image.save('generated_image.jpg')")

## Part 3: Audio Processing

Use `dspy.Audio` for speech and audio understanding tasks.

In [ ]:
# Example 3.1: Audio Transcription
print_step("Audio Transcription", "Converting speech to text")

class AudioTranscriber(dspy.Signature):
    """Transcribe audio to text."""
    audio = dspy.InputField(desc="Audio file to transcribe")
    transcript = dspy.OutputField(desc="The transcribed text")

transcriber = dspy.Predict(AudioTranscriber)

print("Use case: Convert speech to text")
print("Example:")
print("  audio = dspy.Audio(path='speech.mp3')")
print("  result = transcriber(audio=audio)")
print("  print(f'Transcript: {result.transcript}')")

In [ ]:
# Example 3.2: Audio Content Analysis
print_step("Audio Analysis", "Analyzing audio content for insights")

class AudioAnalyzer(dspy.Signature):
    """Analyze audio content."""
    audio = dspy.InputField(desc="Audio file to analyze")
    summary = dspy.OutputField(desc="Summary of audio content")
    emotions = dspy.OutputField(desc="Detected emotions or tone")
    key_points = dspy.OutputField(desc="Key points or topics discussed")

analyzer = dspy.Predict(AudioAnalyzer)

print("Use case: Analyze audio content for insights")
print("Example:")
print("  audio = dspy.Audio(path='meeting.mp3')")
print("  result = analyzer(audio=audio)")
print("  print(f'Summary: {result.summary}')")
print("  print(f'Emotions: {result.emotions}')")
print("  print(f'Key points: {result.key_points}')")

## Part 4: Custom Multimodal Modules

Build advanced applications that combine text, images, and audio in a single module.

In [ ]:
class MultimodalAssistant(dspy.Module):
    """A comprehensive assistant that can handle text, images, and audio."""
    def __init__(self):
        super().__init__()

        class AnalyzeImage(dspy.Signature):
            """Analyze an image and extract information."""
            image = dspy.InputField()
            context = dspy.InputField(desc="Additional context or question")
            analysis = dspy.OutputField(desc="Detailed analysis")

        class ProcessAudio(dspy.Signature):
            """Process audio and extract information."""
            audio = dspy.InputField()
            context = dspy.InputField(desc="Additional context")
            information = dspy.OutputField(desc="Extracted information")

        class CombineMultimodal(dspy.Signature):
            """Combine information from multiple modalities."""
            text_input = dspy.InputField(desc="Text information")
            image_info = dspy.InputField(desc="Information from images")
            audio_info = dspy.InputField(desc="Information from audio")
            response = dspy.OutputField(desc="Comprehensive response")

        self.analyze_image = dspy.Predict(AnalyzeImage)
        self.process_audio = dspy.Predict(ProcessAudio)
        self.combine = dspy.ChainOfThought(CombineMultimodal)

    def forward(self, text, image=None, audio=None):
        image_info = "No image provided"
        audio_info = "No audio provided"

        if image:
            img_result = self.analyze_image(image=image, context=text)
            image_info = img_result.analysis

        if audio:
            aud_result = self.process_audio(audio=audio, context=text)
            audio_info = aud_result.information

        final_result = self.combine(
            text_input=text,
            image_info=image_info,
            audio_info=audio_info
        )

        return dspy.Prediction(
            response=final_result.response,
            reasoning=final_result.reasoning,
            image_analysis=image_info,
            audio_analysis=audio_info
        )

assistant = MultimodalAssistant()

print("MultimodalAssistant capabilities:")
print("  - Processing images with context")
print("  - Processing audio with context")
print("  - Combining multimodal information")
print("  - Generating comprehensive responses")
print("\nExample usage:")
print("  img = dspy.Image(path='product.jpg')")
print("  result = assistant(text='What are the key features?', image=img)")
print("  print(result.response)")

## Part 5: Practical Tips

Best practices for multimodal DSPy applications.

In [ ]:
print("1. Loading Multimodal Data:")
print("   - From file: dspy.Image(path='image.jpg')")
print("   - From URL: dspy.Image(url='https://example.com/image.jpg')")
print("   - From bytes: dspy.Image(data=image_bytes)")
print()
print("2. Supported Formats:")
print("   - Images: JPEG, PNG, WebP, GIF")
print("   - Audio: MP3, WAV, OGG, M4A")
print()
print("3. Model Compatibility:")
print("   - Vision: GPT-4o, GPT-4 Turbo, Claude 3+, Gemini 1.5+")
print("   - Audio: Gemini 1.5+, Whisper (via OpenAI)")
print()
print("4. Best Practices:")
print("   - Compress large images before sending")
print("   - Provide context with multimodal inputs")
print("   - Handle missing modalities gracefully")
print("   - Test with different model providers")
print()
print("5. Common Use Cases:")
print("   - Document analysis (OCR, layout understanding)")
print("   - Product catalog generation")
print("   - Accessibility (image descriptions for screen readers)")
print("   - Content moderation")
print("   - Audio transcription and analysis")

## Summary

**Key Takeaways:**
1. DSPy 3.x provides first-class support for multimodal AI
2. Use `dspy.Image` for vision tasks (description, VQA, comparison)
3. Use `dspy.Audio` for audio tasks (transcription, analysis)
4. Multimodal fields work seamlessly with DSPy signatures
5. Combine modalities in custom modules for powerful applications
6. Always check model compatibility for multimodal features